<a href="https://colab.research.google.com/github/zemenfes-afk/SKY-Innovator/blob/main/Sky_innovater_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part Start with satelite images and mask and build model

# STEP 1: Download Raw Data & Install Converter Tools

In [ ]:
# 1. Install essential tools for downloading and converting
!pip install -q kagglehub tifffile imagecodecs

import kagglehub #needed downloading  dataset from kaggle
import os
import shutil  #Copying, moving, deleting files and directories and handle zip folders
import tifffile as tiff  #TIFF image files
import numpy as np       #handle matmatical operations
from PIL import Image    #handle resizing and cropiing images
from tqdm.notebook import tqdm # For nice progress bars or shows the progress

# 2. Download Raw Data
print("⬇️ Checking for raw dataset...")
raw_path = kagglehub.dataset_download("catiowiec/amazon-and-atlantic-forest-sentinel-2-multiband")
print(f"✅ Raw data located at: {raw_path}")

# 3. Define where we want our nice, clean Amazon JPGs to go
Clean_Amazon_Dir = "Clean_Amazon_Dataset"
Image_Dir = os.path.join(Clean_Amazon_Dir, "images")
Mask_Dir = os.path.join(Clean_Amazon_Dir, "masks")

# Clean up previous runs if they exist
if os.path.exists(Clean_Amazon_Dir): shutil.rmtree(Clean_Amazon_Dir)
os.makedirs(Image_Dir, exist_ok=True)
os.makedirs(Mask_Dir, exist_ok=True)
print("✅ Created clean output folders.")

# STEP 2: The Great Converter Script (TIF $\rightarrow$ JPG)

In [ ]:
# ==========================================
# STEP 2: CONVERT TIF TO JPG (ROBUST MODE)
# ==========================================
import tifffile as tiff
import numpy as np
from PIL import Image
import os
from tqdm.notebook import tqdm

# --- FIX: DEFINE THE MISSING VARIABLE HERE ---
amazon_sources = [
    os.path.join(raw_path, "AMAZON/AMAZON/Training"),
    os.path.join(raw_path, "AMAZON/AMAZON/Validation")
]
# ---------------------------------------------

print("🔄 Starting Conversion Process (Attempt 2 - Robust Mode)...")
converted_count = 0

for source_folder in amazon_sources:
    source_img_path = os.path.join(source_folder, "image")
    source_mask_path = os.path.join(source_folder, "label")

    if not os.path.exists(source_img_path): continue

    tif_files = [f for f in os.listdir(source_img_path) if f.endswith('.tif')]

    for tif_name in tqdm(tif_files):
        tif_img_full_path = os.path.join(source_img_path, tif_name)
        tif_mask_full_path = os.path.join(source_mask_path, tif_name)

        if os.path.exists(tif_mask_full_path):
            try:
                # 1. Read TIF
                img_data = tiff.imread(tif_img_full_path)

                # 2. INTELLIGENT SHAPE CORRECTION
                # Fix "Barcode Noise" by flipping dimensions if needed
                if img_data.shape[0] < 20 and img_data.ndim == 3:  #checking for the image on order of [3,256,256] or reversed
                     img_data = np.moveaxis(img_data, 0, -1)

                # Keep only RGB (First 3 channels)
                if img_data.shape[-1] > 3:
                    img_data = img_data[:, :, 0:3]

                # 3. Brightness Normalization (Fix Dark Images)
                max_val = img_data.max()
                if max_val > 0:
                    p98 = np.percentile(img_data, 98)
                    img_data = np.clip(img_data, 0, p98)
                    img_data = (img_data / p98 * 255).astype(np.uint8)
                else:
                    img_data = img_data.astype(np.uint8)

                # 4. Save as JPG
                new_img_name = tif_name.replace('.tif', '.jpg')
                Image.fromarray(img_data).save(os.path.join(Image_Dir, new_img_name), quality=95)

                # 5. Save Mask (as PNG)
                mask_data = Image.open(tif_mask_full_path).convert("L")
                new_mask_name = tif_name.replace('.tif', '.png')
                mask_data.save(os.path.join(Mask_Dir, new_mask_name))

                converted_count += 1

            except Exception as e:
                pass # Skip broken files

print(f"\n🎉 SUCCESS! Converted {converted_count} images.")
print(f"Your clean data is in: {Clean_Amazon_Dir}")

# checking How many loaded images

In [ ]:
import os

# 1. Check folder counts
img_count = len(os.listdir("Clean_Amazon_Dataset/images"))
mask_count = len(os.listdir("Clean_Amazon_Dataset/masks"))

print(f"📂 Images found: {img_count}")
print(f"📂 Masks found:  {mask_count}")

# 2. Compare them (They MUST be equal)
if img_count == 0:
    print("❌ ERROR: No images converted! Did you run Step 2 correctly?")
elif img_count != mask_count:
    print(f"⚠️ WARNING: Mismatch! You have {img_count} images but only {mask_count} masks.")
    print("This means some masks failed to convert.")
else:
    print("✅ SUCCESS: You have exactly matching pairs. The folder view might just be lagging!")
    # Show the first few filenames to be sure
    print("\nFirst 3 Mask Files:")
    print(os.listdir("Clean_Amazon_Dataset/masks")[:3])

# STEP 3: Load the dataset or Dataloader FOR IMAGE AND MASKS

In [ ]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import glob
from PIL import Image

class SimpleAmazonDataset(Dataset):
    def __init__(self, root_dir):
        self.img_dir = os.path.join(root_dir, "images")
        self.mask_dir = os.path.join(root_dir, "masks")

        # Find all JPG images
        self.images = sorted(glob.glob(os.path.join(self.img_dir, "*.jpg")))

        # Standard transforms for U-Net
        self.transform = transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]

        # Match image to corresponding mask (.jpg → .png)
        filename = os.path.basename(img_path)
        mask_filename = filename.replace('.jpg', '.png')
        mask_path = os.path.join(self.mask_dir, mask_filename)

        # Load images
        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")  # Grayscale

        # Transform
        image = self.transform(image)
        mask = self.transform(mask)

        # Binarize mask (0 or 1)
        mask = (mask > 0).float().long().squeeze(0)

        return image, mask


# ── DataLoader ───────────────────────────────────────────────────────────────
amazon_dataset = SimpleAmazonDataset(Clean_Amazon_Dir)
train_loader = DataLoader(amazon_dataset, batch_size=16, shuffle=True)

print(f"✅ Simple DataLoader ready with {len(amazon_dataset)} Amazon images.")

# ── Visualise a batch ────────────────────────────────────────────────────────
images, masks = next(iter(train_loader))

plt.figure(figsize=(10, 10))
for i in range(3):
    plt.subplot(3, 2, i*2 + 1)
    plt.imshow(images[i].permute(1, 2, 0))
    plt.title("Converted JPG Image")
    plt.axis("off")

    plt.subplot(3, 2, i*2 + 2)
    plt.imshow(masks[i], cmap='gray')
    plt.title("Corresponding Mask")
    plt.axis("off")

plt.tight_layout()
plt.show()

# Step 4 - Install the SMP used for segementation using Pytorch

SMP is helpfull pretrained Model library specially for U-net

In [ ]:
!pip install segmentation-models-pytorch

# step - 5 MODEL TRAINING

In [ ]:
import torch
import torch.optim as optim
import matplotlib.pyplot as plt
import segmentation_models_pytorch as smp
from segmentation_models_pytorch.losses import DiceLoss, SoftCrossEntropyLoss
import segmentation_models_pytorch.metrics as smp_metrics

# ── 1. MODEL SETUP ────────────────────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️  Using device: {device}")

model = smp.Unet(
    encoder_name="efficientnet-b2",
    encoder_weights="imagenet",
    in_channels=3,
    classes=2  # Background vs Forest
).to(device)

# ── 2. LOSS, OPTIMIZER & SCHEDULER ───────────────────────────────────────────
dice_loss = DiceLoss(mode='multiclass')
ce_loss   = SoftCrossEntropyLoss(smooth_factor=0.1)
optimizer = optim.Adam(model.parameters(), lr=0.0001) #learning rate is 0.0001
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
)                                                  #learning rate scheduled every minimumstage by 0.5 and If loss doesn't improve for 3 epochs, reduce LR

# ── 3. TRAINING LOOP ─────────────────────────────────────────────────────────
epochs       = 30
best_iou     = 0.0
loss_history = []
iou_history  = []

print(f"🚀 Starting Training for {epochs} Epochs...")
print("-" * 65)

model.train()
for epoch in range(epochs):
    epoch_loss = 0
    epoch_iou  = 0

    for images, masks in train_loader:
        images = images.to(device)       #Move data to GPU
        masks  = masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)          #Model predicts segmentation maps.([batch, classes, height, width]) /Forward Pass

        # Combined Dice + CrossEntropy loss
        loss = dice_loss(outputs, masks) + ce_loss(outputs, masks.long())  #Measures how wrong predictions are.
        loss.backward()            #Backward Pass/Backpropagation
        optimizer.step()           #update weights

        epoch_loss += loss.item()

        # IoU per batch          # batch one sample dateset sent
        with torch.no_grad():
            preds = torch.argmax(torch.softmax(outputs, dim=1), dim=1)
            tp, fp, fn, tn = smp_metrics.get_stats(
                preds, masks, mode='multiclass', num_classes=2
            )
            epoch_iou += smp_metrics.iou_score(          #IoU = TP / (TP + FP + FN)
                tp, fp, fn, tn, reduction='macro'
            ).item()

    avg_loss = epoch_loss / len(train_loader)           #average of loss
    avg_iou  = epoch_iou  / len(train_loader)
    loss_history.append(avg_loss)
    iou_history.append(avg_iou)

    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step(avg_loss)

    # Save best model checkpoint
    if avg_iou > best_iou:
        best_iou = avg_iou
        torch.save(model.state_dict(), "/content/best_amazon_model.pth")
        save_msg = "💾 Saved!"
    else:
        save_msg = ""

    print(f"Epoch [{epoch+1:02d}/{epochs}] | Loss: {avg_loss:.4f} | IoU: {avg_iou:.4f} | LR: {current_lr:.6f} | {save_msg}")

print("-" * 65)
print(f"🎉 Training Complete! Best IoU: {best_iou:.4f}")

# ── 4. SAVE FINAL MODEL TO GOOGLE DRIVE (Colab) ───────────────────────────────
torch.save(model.state_dict(), "/content/final_amazon_model.pth")
print("💾 Final model saved → /content/final_amazon_model.pth")

# Optional: save to Drive so it survives session reset
# from google.colab import drive
# drive.mount('/content/drive')
# torch.save(model.state_dict(), "/content/drive/MyDrive/final_amazon_model.pth")

# ── 5. PLOT TRAINING CURVES ───────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(loss_history, marker='o', color='red',   label='Combined Loss')
ax1.set_title("Training Loss (Lower is Better)")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax1.grid(True); ax1.legend()

ax2.plot(iou_history,  marker='o', color='green', label='IoU Score')
ax2.set_title("IoU Score (Higher is Better)")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("IoU")
ax2.axhline(y=0.75, color='orange', linestyle='--', label='Good (0.75)')
ax2.axhline(y=0.85, color='blue',   linestyle='--', label='Excellent (0.85)')
ax2.grid(True); ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

**Step -6 :Visualizing the Model output**

In [ ]:
import matplotlib.pyplot as plt
import torch
import numpy as np

# 1. Switch Model to "Evaluation" Mode (Turns off training layers)
model.eval()

# 2. Get a batch of Validation Data
# We use the 'train_loader' for now since we combined folders
images, masks = next(iter(train_loader))

images = images.to(device)
masks = masks.to(device)

# 3. Ask AI to Predict
with torch.no_grad():
    predictions = model(images)
    # The model outputs "probabilities" (floats).
    # We convert them to Binary (0 or 1) by taking the max value.
    pred_masks = torch.argmax(predictions, dim=1)

# 4. Plot the "Human vs Machine" Comparison
plt.figure(figsize=(12, 12))
for i in range(4): # Show top 4 images
    # A. Original Satellite Image
    img_np = images[i].cpu().permute(1, 2, 0).numpy()

    # B. Ground Truth (What Humans Labeled)
    true_mask = masks[i].cpu().numpy()

    # C. AI Prediction (What your Model saw)
    pred_mask = pred_masks[i].cpu().numpy()

    # Plotting
    plt.subplot(4, 3, i*3 + 1)
    plt.imshow(img_np)
    plt.title("Satellite Input")
    plt.axis("off")

    plt.subplot(4, 3, i*3 + 2)
    plt.imshow(true_mask, cmap='gray')
    plt.title("Ground Truth")
    plt.axis("off")

    plt.subplot(4, 3, i*3 + 3)
    plt.imshow(pred_mask, cmap='jet') # Color map to make it pop
    plt.title("AI Prediction")
    plt.axis("off")

plt.tight_layout()
plt.show()

# Part 2 let' start with the drone images because our target is drone image but take sample drone images and generate their mask using the satelite images."

# Step -1 Checking the drone image size

In [ ]:
# ── STEP 1: Mount Google Drive ─────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── Check your frames are there ────────────────────────────────────────────────
import glob, os
from PIL import Image

# !! Change this to your actual Google Drive folder path !!
DRONE_FRAME_DIR = "/content/drive/MyDrive/forest_frames_wide"   # image of drone
FRAME_MASK_DIR  = "/content/drive/MyDrive/drone_frame_masks"  # masks saved to Drive too
os.makedirs(FRAME_MASK_DIR, exist_ok=True)

frame_paths = sorted(
    glob.glob(os.path.join(DRONE_FRAME_DIR, "*.jpg")) +
    glob.glob(os.path.join(DRONE_FRAME_DIR, "*.png"))
)

print(f"📁 Found {len(frame_paths)} frames in Drive")
print(f"📐 Sample resolutions:")
for p in frame_paths[:5]:
    img = Image.open(p)
    print(f"   {os.path.basename(p):40s} → {img.size[0]}x{img.size[1]}")

# Step - Genrate The MASK of Drone Image M1-M4 for Masking(function of MASK)

In [ ]:
# ── Updated constants — less aggressive ───────────────────────────────────────
WEIGHT_1 = 0.15   # Brightness — reduced (catches too much)
WEIGHT_2 = 0.20   # Texture    — reduced
WEIGHT_3 = 0.45   # Color/Chlorophyll — increased (most reliable)
WEIGHT_4 = 0.20   # Edge density — slightly increased

def generate_combined_forest_mask(image_pil):   #our function for MASK
    img_rgb = np.array(image_pil.convert("RGB"))
    img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    orig_h, orig_w = img_bgr.shape[:2]

    img_hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    H = img_hsv[:, :, 0].astype(np.float32)
    S = img_hsv[:, :, 1].astype(np.float32)
    V = img_hsv[:, :, 2].astype(np.float32)

    # ── Part 1: Strict dead detection — only obvious dead areas ───────────────
    m1 = find_and_plot_contours(img_bgr)
    m2 = analyze_texture_and_saturation(img_bgr)
    m3 = detect_dead_color_foliage(img_bgr)
    m4 = detect_edge_density(img_bgr)

    dead_weighted = (
        m1.astype(float) * WEIGHT_1 +
        m2.astype(float) * WEIGHT_2 +
        m3.astype(float) * WEIGHT_3 +
        m4.astype(float) * WEIGHT_4
    ).astype(np.uint8)

    # ✅ Raised threshold back up — only flag clearly dead areas
    _, dead_mask_small = cv2.threshold(dead_weighted, 160, 255, cv2.THRESH_BINARY)
    dead_mask = cv2.resize(dead_mask_small, (orig_w, orig_h),
                           interpolation=cv2.INTER_NEAREST)

    # ── Part 2: Grey dead — STRICT (only very grey + medium brightness) ────────
    # NOT catching orange/brown autumn — only true grey bare branches
    grey_dead = (
        (S < 25) &          # Very low saturation = truly grey
        (V > 60) &          # Not too dark (shadows)
        (V < 180) &         # Not too bright (sky)
        (H > 15) | (H < 5)  # Not in warm orange hue range
    ).astype(np.uint8) * 255

    # Only very bright washed-out areas (bare branches in sunlight)
    pale_dead = ((S < 15) & (V > 160) & (V < 220)).astype(np.uint8) * 255

    # Combine — require BOTH dead_mask AND (grey OR pale) to flag as dead
    # This prevents over-detection from M1-M4 alone at low resolution
    dead_combined = cv2.bitwise_or(
        dead_mask,
        cv2.bitwise_and(grey_dead, pale_dead)  # Only flag if both agree
    )

    # Clean dead mask
    kernel_d    = np.ones((5, 5), np.uint8)
    dead_clean  = cv2.morphologyEx(dead_combined, cv2.MORPH_OPEN,  kernel_d, iterations=2)
    dead_clean  = cv2.morphologyEx(dead_clean,    cv2.MORPH_CLOSE, kernel_d, iterations=2)

    # ── Part 3: Autumn Forest Detection ───────────────────────────────────────
    green       = (H >= 35) & (H <= 85)  & (S >= 20) & (V >= 15)
    yellow      = (H >= 18) & (H <= 40)  & (S >= 30) & (V >= 40)
    orange      = (H >= 5)  & (H <= 25)  & (S >= 45) & (V >= 25) & (V <= 230)
    brown       = (H >= 3)  & (H <= 20)  & (S >= 35) & (V >= 15) & (V <= 175)
    red         = (((H >= 0) & (H <= 10)) | ((H >= 160) & (H <= 179))) \
                  & (S >= 35) & (V >= 20) & (V <= 200)
    dark_canopy = (V <= 65) & (S >= 25)

    autumn_forest = (
        green | yellow | orange | brown | red | dark_canopy
    ).astype(np.uint8) * 255

    # ── Part 4: Dead WINS over forest ─────────────────────────────────────────
    forest_mask = cv2.bitwise_and(
        autumn_forest,
        cv2.bitwise_not(dead_clean)
    )

    # ── Part 5: Cleanup ────────────────────────────────────────────────────────
    kernel  = np.ones((5, 5), np.uint8)
    closed  = cv2.morphologyEx(forest_mask, cv2.MORPH_CLOSE, kernel, iterations=3)
    opened  = cv2.morphologyEx(closed,      cv2.MORPH_OPEN,  kernel, iterations=2)
    cleaned = cv2.medianBlur(opened, 5)

    final = np.where(cleaned > 0, 0, 255).astype(np.uint8)
    return Image.fromarray(final), dead_clean


# ── Test on 3 frames ──────────────────────────────────────────────────────────
test_samples = random.sample(frame_paths, 3)
fig, axes = plt.subplots(3, 4, figsize=(18, 11))

for i, img_path in enumerate(test_samples):
    image = Image.open(img_path).convert("RGB")
    mask, dead_viz = generate_combined_forest_mask(image)

    img_np   = np.array(image.resize((480, 270))).astype(np.float32)
    mask_np  = np.array(mask.resize((480, 270)))
    dead_np  = cv2.resize(dead_viz, (480, 270), interpolation=cv2.INTER_NEAREST)

    overlay  = img_np.copy()
    overlay[mask_np == 0]   = (img_np[mask_np == 0]   * 0.5 + np.array([0,   180, 0]) * 0.5)
    overlay[mask_np == 255] = (img_np[mask_np == 255] * 0.5 + np.array([180, 0,   0]) * 0.5)
    overlay  = overlay.astype(np.uint8)

    forest_pct = (mask_np == 0).sum() / mask_np.size * 100
    dead_pct   = (dead_np  > 0).sum() / dead_np.size * 100

    axes[i][0].imshow(image.resize((480, 270)))
    axes[i][0].set_title(f"Original\n{os.path.basename(img_path)[:18]}", fontsize=8)
    axes[i][0].axis("off")

    axes[i][1].imshow(dead_np, cmap='hot')
    axes[i][1].set_title(f"Dead Detection\n{dead_pct:.1f}% flagged", fontsize=8)
    axes[i][1].axis("off")

    axes[i][2].imshow(mask.resize((480, 270)), cmap='gray')
    axes[i][2].set_title(f"Final Mask\n🌲 {forest_pct:.1f}%", fontsize=8)
    axes[i][2].axis("off")

    axes[i][3].imshow(overlay)
    axes[i][3].set_title("🟢 Forest | 🔴 Dead/Non-Forest", fontsize=8)
    axes[i][3].axis("off")

plt.suptitle("V4 Balanced — Strict Dead Detection", fontsize=13)
plt.tight_layout()
plt.show()

#Step-2 after we build the masking on the above now let' generate the full 392 image

In [ ]:
from tqdm import tqdm
import glob, os
from PIL import Image

FRAME_MASK_DIR = "/content/drive/MyDrive/drone_frame_masks_v4"
os.makedirs(FRAME_MASK_DIR, exist_ok=True)

print(f"🍂 Generating final masks for {len(frame_paths)} frames...")
print(f"   Saving → {FRAME_MASK_DIR}\n")

failed = []
for img_path in tqdm(frame_paths, desc="Masking"):
    try:
        filename  = os.path.basename(img_path)
        mask_name = filename.rsplit('.', 1)[0] + '.png'
        mask_path = os.path.join(FRAME_MASK_DIR, mask_name)

        if os.path.exists(mask_path):  # Skip already done
            continue

        image     = Image.open(img_path).convert("RGB")
        mask, _   = generate_combined_forest_mask(image)     # method of generate_combined_forest_mask call
        mask.save(mask_path)

    except Exception as e:
        failed.append(os.path.basename(img_path))
        print(f"⚠️ Failed: {os.path.basename(img_path)} → {e}")

total = len(glob.glob(os.path.join(FRAME_MASK_DIR, "*.png")))
print(f"\n✅ Done! {total} / {len(frame_paths)} masks saved")
if failed:
    print(f"⚠️  Failed: {len(failed)} → {failed}")

# Step 2 — Auto-Generate Masks Using Your Satellite Model

In [ ]:
import torch
import torch.optim as optim
import matplotlib.pyplot as plt
import segmentation_models_pytorch as smp
from segmentation_models_pytorch.losses import DiceLoss, SoftCrossEntropyLoss
import segmentation_models_pytorch.metrics as smp_metrics

# ── CONFIG ────────────────────────────────────────────────────────────────────
BEST_AUTUMN_MODEL  = "/content/best_amazon_model.pth"
FINAL_AUTUMN_MODEL = "/content/final_autumn_model.pth"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️  Using device: {device}")

# ── LOAD MODEL — start from best checkpoint ───────────────────────────────────
model = smp.Unet(
    encoder_name="efficientnet-b2",
    encoder_weights=None,
    in_channels=3,
    classes=2
).to(device)

model.load_state_dict(torch.load(BEST_AUTUMN_MODEL, map_location=device))
print(f"✅ Loaded best_autumn_model.pth (IoU: 0.6748)")

# ── UNFREEZE ENCODER — same as satellite training ─────────────────────────────
for param in model.encoder.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"🔓 Full model trainable: {trainable:,} / {total:,} params")

# ── SAME LOSS AS SATELLITE — simpler is better ────────────────────────────────
dice_loss = DiceLoss(mode='multiclass')
ce_loss   = SoftCrossEntropyLoss(smooth_factor=0.1)

# ── SAME OPTIMIZER STYLE AS SATELLITE ────────────────────────────────────────
# Higher LR than fine-tune but lower than satellite
optimizer = optim.Adam(model.parameters(), lr=0.00005)

# ── SAME SCHEDULER AS SATELLITE — ReduceLROnPlateau ──────────────────────────
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
)

# ── TRAINING LOOP — identical structure to satellite ─────────────────────────
epochs    = 60
best_iou  = 0.6748   # Start from previous best
loss_history = []
iou_history  = []

print(f"\n🚀 Continued Training for {epochs} Epochs...")
print("-" * 65)

model.train()
for epoch in range(epochs):
    epoch_loss = 0
    epoch_iou  = 0

    for images, masks in autumn_loader:
        images = images.to(device)
        masks  = masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)

        # Same loss combo as satellite training
        loss = dice_loss(outputs, masks) + ce_loss(outputs, masks.long())
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

        with torch.no_grad():
            preds = torch.argmax(torch.softmax(outputs, dim=1), dim=1)
            tp, fp, fn, tn = smp_metrics.get_stats(
                preds, masks, mode='multiclass', num_classes=2
            )
            epoch_iou += smp_metrics.iou_score(
                tp, fp, fn, tn, reduction='macro'
            ).item()

    avg_loss = epoch_loss / len(autumn_loader)
    avg_iou  = epoch_iou  / len(autumn_loader)
    loss_history.append(avg_loss)
    iou_history.append(avg_iou)

    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step(avg_loss)   # Same as satellite — monitor loss

    if avg_iou > best_iou:
        best_iou = avg_iou
        torch.save(model.state_dict(), BEST_AUTUMN_MODEL)
        save_msg = "💾 Saved!"
    else:
        save_msg = ""

    print(
        f"Epoch [{epoch+1:02d}/{epochs}] | Loss: {avg_loss:.4f} | "
        f"IoU: {avg_iou:.4f} | LR: {current_lr:.6f} | {save_msg}"
    )

print("-" * 65)
print(f"🎉 Training Complete! Best IoU: {best_iou:.4f}")

# ── SAVE ──────────────────────────────────────────────────────────────────────
torch.save(model.state_dict(), FINAL_AUTUMN_MODEL)
print(f"💾 Final model → {FINAL_AUTUMN_MODEL}")

#Save to Drive
torch.save(model.state_dict(), "/content/drive/MyDrive/final_autumn_model.pth")

# ── PLOT ──────────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(loss_history, marker='o', color='red', label='Combined Loss')
ax1.set_title("Training Loss (Lower is Better)")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax1.grid(True); ax1.legend()

ax2.plot(iou_history, marker='o', color='green', label='IoU Score')
ax2.axhline(y=0.75, color='orange', linestyle='--', label='Good (0.75)')
ax2.axhline(y=0.85, color='blue',   linestyle='--', label='Excellent (0.85)')
ax2.set_title("IoU Score (Higher is Better)")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("IoU")
ax2.grid(True); ax2.legend()

plt.suptitle(
    f"Autumn Continued Training | Best IoU: {best_iou:.4f} | "
    f"Starting from: 0.89",
    fontsize=12
)
plt.tight_layout()
plt.show()

# Step 4- let's test our model

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torchvision.transforms as transforms
import segmentation_models_pytorch as smp
from PIL import Image
import cv2
import os
import glob

# ── 1. LOAD MODEL ─────────────────────────────────────────────────────────────
MODEL_PATH = "/content/final_autumn_model.pth"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️  Device: {device}")

model = smp.Unet(
    encoder_name="efficientnet-b2",
    encoder_weights=None,
    in_channels=3,
    classes=2
).to(device)

model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()
print(f"✅ Model loaded — IoU: 0.9014")

# ── 2. TRANSFORM ──────────────────────────────────────────────────────────────
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ── 3. PREDICT SINGLE IMAGE ───────────────────────────────────────────────────
def predict_image(image_pil, model, device):
    """
    Input:  PIL image (any size)
    Output: prediction mask (0=Forest, 1=Non-Forest)
            resized back to original dimensions
    """
    orig_w, orig_h = image_pil.size

    inp = transform(image_pil).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(inp)
        probs  = torch.softmax(output, dim=1)
        pred   = torch.argmax(probs, dim=1)

    pred_np = pred.squeeze().cpu().numpy().astype(np.uint8)

    # Resize back to original
    pred_resized = cv2.resize(
        pred_np, (orig_w, orig_h),
        interpolation=cv2.INTER_NEAREST
    )

    # Confidence map (forest probability)
    forest_prob = probs[0, 0].cpu().numpy()
    forest_prob = cv2.resize(forest_prob, (orig_w, orig_h))

    return pred_resized, forest_prob


# ── 4. FULL VISUALIZATION ─────────────────────────────────────────────────────
def visualize_prediction(image_pil, pred_mask, forest_prob, title=""):
    """
    4-panel visualization:
    1. Original image
    2. Prediction mask (B&W)
    3. Color overlay
    4. Confidence heatmap
    """
    img_np    = np.array(image_pil)
    img_float = img_np.astype(np.float32)

    # ── Color overlay ──────────────────────────────────────────────────────────
    overlay = img_float.copy()
    overlay[pred_mask == 0] = (
        img_float[pred_mask == 0] * 0.45 +
        np.array([34, 139, 34]) * 0.55     # Forest green
    )
    overlay[pred_mask == 1] = (
        img_float[pred_mask == 1] * 0.45 +
        np.array([180, 30, 30]) * 0.55     # Dead red
    )
    overlay = np.clip(overlay, 0, 255).astype(np.uint8)

    # ── Stats ──────────────────────────────────────────────────────────────────
    total_px      = pred_mask.size
    forest_px     = (pred_mask == 0).sum()
    nonforest_px  = (pred_mask == 1).sum()
    forest_pct    = forest_px    / total_px * 100
    nonforest_pct = nonforest_px / total_px * 100

    # ── Plot ───────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    # Panel 1 — Original
    axes[0].imshow(img_np)
    axes[0].set_title("Original Image", fontsize=11, fontweight='bold')
    axes[0].axis("off")

    # Panel 2 — B&W Mask
    axes[1].imshow(pred_mask, cmap='gray')
    axes[1].set_title(
        f"Prediction Mask\nBlack=Forest | White=Non-Forest",
        fontsize=11, fontweight='bold'
    )
    axes[1].axis("off")

    # Panel 3 — Color Overlay
    axes[2].imshow(overlay)
    axes[2].set_title(
        f"Vegetation Map\n🌲 {forest_pct:.1f}%  |  🟫 {nonforest_pct:.1f}%",
        fontsize=11, fontweight='bold'
    )
    # Legend
    forest_patch    = mpatches.Patch(color=(34/255, 139/255, 34/255), label=f'Forest {forest_pct:.1f}%')
    nonforest_patch = mpatches.Patch(color=(180/255, 30/255, 30/255),  label=f'Non-Forest {nonforest_pct:.1f}%')
    axes[2].legend(handles=[forest_patch, nonforest_patch],
                   loc='lower right', fontsize=8)
    axes[2].axis("off")

    # Panel 4 — Confidence Heatmap
    heatmap = axes[3].imshow(forest_prob, cmap='RdYlGn', vmin=0, vmax=1)
    axes[3].set_title(
        "Forest Confidence\nGreen=Sure Forest | Red=Sure Non-Forest",
        fontsize=11, fontweight='bold'
    )
    axes[3].axis("off")
    plt.colorbar(heatmap, ax=axes[3], fraction=0.046, pad=0.04)

    if title:
        fig.suptitle(title, fontsize=13, y=1.02)

    plt.tight_layout()
    plt.show()

    return forest_pct, nonforest_pct


# ── 5. TEST ON YOUR DRONE FRAMES ──────────────────────────────────────────────
DRONE_FRAME_DIR = "/content/drive/MyDrive/forest_frames_wide"  # ← your frames

frame_paths = sorted(
    glob.glob(os.path.join(DRONE_FRAME_DIR, "*.jpg")) +
    glob.glob(os.path.join(DRONE_FRAME_DIR, "*.JPG")) +
    glob.glob(os.path.join(DRONE_FRAME_DIR, "*.png"))
)

# Test on 6 random frames
import random
test_paths = random.sample(frame_paths, min(6, len(frame_paths)))

print(f"\n🔍 Testing on {len(test_paths)} drone frames...\n")

results = []
for img_path in test_paths:
    filename = os.path.basename(img_path)
    image    = Image.open(img_path).convert("RGB")

    pred_mask, forest_prob = predict_image(image, model, device)
    forest_pct, nonforest_pct = visualize_prediction(
        image, pred_mask, forest_prob,
        title=f"🌿 Vegetation Map — {filename}"
    )

    results.append({
        "file":        filename,
        "forest_pct":  forest_pct,
        "nonforest_pct": nonforest_pct
    })
    print(f"  ✅ {filename:40s} | 🌲 Forest: {forest_pct:.1f}% | 🟫 Non-Forest: {nonforest_pct:.1f}%")


# ── 6. SUMMARY TABLE ──────────────────────────────────────────────────────────
print(f"\n{'─'*65}")
print(f"📊 VEGETATION MAPPING SUMMARY")
print(f"{'─'*65}")
print(f"  {'Filename':<40} {'Forest':>8} {'Non-Forest':>12}")
print(f"{'─'*65}")

for r in results:
    print(f"  {r['file']:<40} {r['forest_pct']:>7.1f}% {r['nonforest_pct']:>11.1f}%")

avg_forest = sum(r['forest_pct'] for r in results) / len(results)
print(f"{'─'*65}")
print(f"  {'AVERAGE':<40} {avg_forest:>7.1f}% {100-avg_forest:>11.1f}%")
print(f"{'─'*65}")

# ── 7. ALSO TEST ON YOUR 5 GREEN DRONE IMAGES ─────────────────────────────────
print(f"\n🌿 Testing on clean drone images (green forest)...")

CLEAN_DRONE_DIR = "/content/drive/MyDrive/dron images"
clean_paths     = sorted(glob.glob(os.path.join(CLEAN_DRONE_DIR, "*.jpg")))

if clean_paths:
    for img_path in clean_paths[:3]:
        filename = os.path.basename(img_path)
        image    = Image.open(img_path).convert("RGB")

        pred_mask, forest_prob = predict_image(image, model, device)
        forest_pct, _ = visualize_prediction(
            image, pred_mask, forest_prob,
            title=f"🌿 Green Drone Image — {filename}"
        )
        print(f"  ✅ {filename:40s} | 🌲 Forest: {forest_pct:.1f}%")
else:
    print("  ⚠️ No clean drone images found — skipping")

## How to handle the backlight and we use the (Shadow-Proof CLAHE) other with dark images and also other 3 class

Metric,Before (Standard Code),After (Shadow-Proof CLAHE),Improvement
Healthy Forest,0.1%,28.7%,+28.6%
Degraded Forest,0.0%,18.6%,+18.6%
Total Vegetation,0.1%,47.3%,🚀 470x Better